In [3]:
from typing import TypedDict

from langgraph.graph import StateGraph,START,END
from langgraph.types import RetryPolicy
from loguru import  logger
from requests import HTTPError


#1. 声明状态
class EmptyState(TypedDict):
    pass
#2. 声明节点
def node_a(state:EmptyState) -> EmptyState:
    logger.info("node a正在运行")
    raise HTTPError

#3. 构建图
builder = StateGraph(state_schema = EmptyState)
builder.add_node(
    "node_a",
    node_a,
    retry_policy=RetryPolicy(
        max_attempts=3,
        jitter = False
    ))

builder.add_edge(START,"node_a")
builder.add_edge("node_a",END)
graph = builder.compile()

try:
    graph.invoke({})
except HTTPError as e:
    logger.info("重试次数耗尽:{}",e)

raw_mermaid = graph.get_graph().draw_mermaid()
print(raw_mermaid)

2026-08-12 23:42:35.910 | INFO     | __main__:node_a:14 - node a正在运行
2026-08-12 23:42:36.416 | INFO     | __main__:node_a:14 - node a正在运行
2026-08-12 23:42:37.421 | INFO     | __main__:node_a:14 - node a正在运行
2026-08-12 23:42:37.422 | INFO     | __main__:<module>:34 - 重试次数耗尽:


---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	node_a(node_a)
	__end__([<p>__end__</p>]):::last
	__start__ --> node_a;
	node_a --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc

